# 01 — Preprocesamiento
Carga del dataset Sentiment140 desde HuggingFace, análisis exploratorio, pipeline de preprocesamiento con spaCy/NLTK y ablación de componentes de limpieza.

In [ ]:
import re
import os
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import emoji
import spacy
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
from datasets import load_dataset
import mlflow
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

load_dotenv()

RANDOM_SEED = int(os.getenv('RANDOM_SEED', 42))
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5000')
MLFLOW_EXPERIMENT_NAME = os.getenv('MLFLOW_EXPERIMENT_NAME', 'sentiment140')

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

np.random.seed(RANDOM_SEED)

DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# NLTK resources
for resource in ['punkt', 'stopwords', 'wordnet', 'punkt_tab']:
    try:
        nltk.data.find(f'tokenizers/{resource}' if 'punkt' in resource else f'corpora/{resource}')
    except LookupError:
        nltk.download(resource, quiet=True)

STOP_WORDS = set(stopwords.words('english'))
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

print('Setup complete — seed:', RANDOM_SEED)

## Sección 1 — Carga de datos

In [ ]:
raw_train_path = DATA_RAW / 'train.csv'
raw_test_path  = DATA_RAW / 'test.csv'

if raw_train_path.exists() and raw_test_path.exists():
    df_train = pd.read_csv(raw_train_path)
    df_test  = pd.read_csv(raw_test_path)
    print('Loaded from local cache.')
else:
    print('Downloading from HuggingFace...')
    ds = load_dataset('adilbekovich/Sentiment140Twitter')
    df_train = ds['train'].to_pandas()
    df_test  = ds['test'].to_pandas()
    df_train.to_csv(raw_train_path, index=False)
    df_test.to_csv(raw_test_path,  index=False)
    print('Saved to data/raw/')

print(f'Train: {len(df_train):,} rows | Test: {len(df_test):,} rows')
df_train.head(3)

In [ ]:
# Normalize column names
text_col   = [c for c in df_train.columns if 'text' in c.lower()][0]
label_col  = [c for c in df_train.columns if 'label' in c.lower() or 'sentiment' in c.lower() or 'target' in c.lower()][0]
print(f'Text column: "{text_col}" | Label column: "{label_col}"')

df_train = df_train[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'label'})
df_test  = df_test[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'label'})

# Sentiment140 uses 0=negative, 4=positive → remap to 0/1
if df_train['label'].max() == 4:
    df_train['label'] = (df_train['label'] == 4).astype(int)
    df_test['label']  = (df_test['label']  == 4).astype(int)

print('Label distribution (train):')
print(df_train['label'].value_counts())

In [ ]:
# === Text statistics (skill NLP pattern) ===
from collections import Counter

all_tokens = ' '.join(df_train['text'].astype(str)).lower().split()
vocab = Counter(all_tokens)
tweet_lengths = df_train['text'].astype(str).apply(lambda x: len(x.split()))

print(f'Vocabulary size (raw): {len(vocab):,}')
print(f'Average tweet length:  {tweet_lengths.mean():.1f} words')
print(f'Median tweet length:   {tweet_lengths.median():.0f} words')
print(f'Max tweet length:      {tweet_lengths.max()} words')
print()
print('Top 10 most common words:')
for word, count in vocab.most_common(10):
    print(f'  {word}: {count:,}')

## Sección 2 — Pipeline de preprocesamiento

In [ ]:
def normalize_elongations(text: str) -> str:
    """goood -> good, looove -> love (max 2 consecutive repeated chars)"""
    return re.sub(r'(.)\1{2,}', r'\1\1', text)


def preprocess_text(
    text: str,
    remove_stopwords: bool = True,
    lemmatize: bool = True,
    emoji_mode: str = 'drop',       # 'keep' | 'translate' | 'drop'
    normalize_elongations_flag: bool = True,
    remove_punct: bool = True,
) -> str:
    """Full preprocessing pipeline: normalize → emoji → tokenize (spaCy) → lemmatize → filter."""
    text = str(text).lower()

    # Remove URLs, mentions, hashtag symbol (keep word)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)

    # Emojis
    if emoji_mode == 'translate':
        text = emoji.demojize(text, delimiters=(' ', ' '))
    elif emoji_mode == 'drop':
        text = emoji.replace_emoji(text, replace='')

    # Normalize elongations
    if normalize_elongations_flag:
        text = normalize_elongations(text)

    # spaCy pipeline: tokenize → (lemmatize) → filter
    doc = nlp(text)
    tokens = []
    for token in doc:
        if remove_punct and (token.is_punct or token.is_space):
            continue
        word = token.lemma_ if lemmatize else token.text
        word = word.strip()
        if not word:
            continue
        if remove_stopwords and word in STOP_WORDS:
            continue
        tokens.append(word)

    return ' '.join(tokens)


sample = df_train['text'].iloc[0]
print('Original:', sample)
print('Processed:', preprocess_text(sample))

## Sección 3 — Ablación de preprocesamiento

Cada configuración se evalúa con BoW + Multinomial NB en un 10% del train (para velocidad) y se registra en MLflow.

In [ ]:
from sklearn.pipeline import Pipeline as SkPipeline

# Use 10% of train for fast ablation
df_ablation, _ = train_test_split(
    df_train, train_size=0.10, random_state=RANDOM_SEED, stratify=df_train['label']
)
X_abl = df_ablation['text'].values
y_abl = df_ablation['label'].values

X_tr, X_val, y_tr, y_val = train_test_split(
    X_abl, y_abl, test_size=0.2, random_state=RANDOM_SEED, stratify=y_abl
)

print(f'Ablation train: {len(X_tr):,} | val: {len(X_val):,}')


def ablation_run(config: dict, member: str = 'nicolas') -> float:
    """Preprocess, fit BoW+NB, log to MLflow, return val F1."""
    X_tr_proc  = [preprocess_text(t, **config) for t in X_tr]
    X_val_proc = [preprocess_text(t, **config) for t in X_val]

    pipe = SkPipeline([
        ('bow', CountVectorizer(max_features=30000)),
        ('clf', MultinomialNB()),
    ])
    pipe.fit(X_tr_proc, y_tr)
    preds = pipe.predict(X_val_proc)
    f1 = f1_score(y_val, preds, average='macro')

    with mlflow.start_run(run_name=f"preprocessing_{config.get('name', 'exp')}"):
        mlflow.set_tag('member', member)
        mlflow.set_tag('stage', 'preprocessing_ablation')
        mlflow.log_params({k: v for k, v in config.items() if k != 'name'})
        mlflow.log_metric('val_f1_macro', f1)

    return f1

In [ ]:
experiments = [
    # Baseline
    dict(name='baseline',
         remove_stopwords=False, lemmatize=False,
         emoji_mode='drop', normalize_elongations_flag=False, remove_punct=True),
    # + Lemmatization
    dict(name='lemmatize_on',
         remove_stopwords=False, lemmatize=True,
         emoji_mode='drop', normalize_elongations_flag=False, remove_punct=True),
    # + Stopwords removal
    dict(name='stopwords_drop',
         remove_stopwords=True, lemmatize=True,
         emoji_mode='drop', normalize_elongations_flag=False, remove_punct=True),
    # + Emoji translate
    dict(name='emoji_translate',
         remove_stopwords=True, lemmatize=True,
         emoji_mode='translate', normalize_elongations_flag=False, remove_punct=True),
    # + Elongation normalization
    dict(name='elongation_on',
         remove_stopwords=True, lemmatize=True,
         emoji_mode='translate', normalize_elongations_flag=True, remove_punct=True),
    # Keep punctuation
    dict(name='keep_punct',
         remove_stopwords=True, lemmatize=True,
         emoji_mode='translate', normalize_elongations_flag=True, remove_punct=False),
]

results = []
for cfg in experiments:
    name = cfg['name']
    print(f'Running: {name} ...', end=' ')
    f1 = ablation_run({k: v for k, v in cfg.items()}, member='nicolas')
    results.append({'config': name, 'val_f1_macro': round(f1, 4)})
    print(f'F1={f1:.4f}')

results_df = pd.DataFrame(results).sort_values('val_f1_macro', ascending=False)
print('\n=== Ablation Results ===')
print(results_df.to_string(index=False))

In [ ]:
# Best config
best_config_name = results_df.iloc[0]['config']
best_cfg_full = next(c for c in experiments if c['name'] == best_config_name)
BEST_PREPROC_CONFIG = {k: v for k, v in best_cfg_full.items() if k != 'name'}
print('Best preprocessing config:', best_config_name)
print(BEST_PREPROC_CONFIG)

In [ ]:
# Apply best config to full train and test
print('Preprocessing full train set...')
df_train['text_clean'] = [preprocess_text(t, **BEST_PREPROC_CONFIG) for t in df_train['text']]
print('Preprocessing full test set...')
df_test['text_clean']  = [preprocess_text(t, **BEST_PREPROC_CONFIG) for t in df_test['text']]

df_train.to_parquet(DATA_PROCESSED / 'train_clean.parquet', index=False)
df_test.to_parquet(DATA_PROCESSED / 'test_clean.parquet',   index=False)

# Save config for downstream notebooks
import json
(DATA_PROCESSED / 'best_preproc_config.json').write_text(
    json.dumps(BEST_PREPROC_CONFIG, indent=2)
)
print('Saved to data/processed/')

## Sección 4 — Visualizaciones (patrón skill NLP)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Sentiment140 — EDA & Preprocessing', fontsize=14, fontweight='bold')

# 1. Top 15 words after preprocessing
clean_tokens = ' '.join(df_train['text_clean'].astype(str)).split()
clean_vocab = Counter(clean_tokens)
top_words = clean_vocab.most_common(15)
words, freqs = zip(*top_words)
axes[0, 0].barh(range(len(words)), freqs, color='steelblue')
axes[0, 0].set_yticks(range(len(words)))
axes[0, 0].set_yticklabels(words)
axes[0, 0].set_xlabel('Frequency')
axes[0, 0].set_title('Top 15 Words (after preprocessing)')
axes[0, 0].invert_yaxis()

# 2. Sentiment distribution
label_counts = df_train['label'].value_counts()
label_names = ['Negative (0)', 'Positive (1)']
axes[0, 1].pie(
    [label_counts.get(0, 0), label_counts.get(1, 0)],
    labels=label_names,
    autopct='%1.1f%%',
    colors=['#e74c3c', '#2ecc71'],
)
axes[0, 1].set_title('Sentiment Distribution (train)')

# 3. Tweet length distribution (raw)
raw_lengths = df_train['text'].astype(str).apply(lambda x: len(x.split()))
axes[1, 0].hist(raw_lengths, bins=40, color='coral', edgecolor='white')
axes[1, 0].set_xlabel('Number of Words')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Tweet Length Distribution (raw)')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Cosine similarity heatmap (sample of 200 docs)
sample_texts = df_train['text_clean'].sample(200, random_state=RANDOM_SEED).tolist()
vec = CountVectorizer(max_features=500)
emb = vec.fit_transform(sample_texts).toarray()
sim_matrix = cosine_similarity(emb)
# Show 10x10 block for readability
im = axes[1, 1].imshow(sim_matrix[:10, :10], cmap='YlOrRd', aspect='auto')
axes[1, 1].set_title('Cosine Similarity (sample 10×10)')
axes[1, 1].set_xlabel('Tweet index')
axes[1, 1].set_ylabel('Tweet index')
plt.colorbar(im, ax=axes[1, 1])

plt.tight_layout()
plt.savefig('../data/processed/eda_visualization.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved eda_visualization.png')

In [ ]:
# Ablation bar chart
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(results_df['config'], results_df['val_f1_macro'], color='steelblue')
ax.bar_label(bars, fmt='%.4f', padding=3)
ax.set_xlabel('F1-score macro (validation)')
ax.set_title('Preprocessing Ablation — F1 by Configuration')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../data/processed/ablation_preprocessing.png', dpi=120, bbox_inches='tight')
plt.show()

## Resumen

- Dataset cargado y guardado en `data/raw/`
- Mejor configuración de preprocesamiento guardada en `data/processed/best_preproc_config.json`
- Datos procesados guardados en `data/processed/train_clean.parquet` y `test_clean.parquet`
- Todos los experimentos registrados en MLflow bajo el experimento `sentiment140`